<a href="https://colab.research.google.com/github/Zinc-zn/MachineLearning2026/blob/main/JS09/JS09_Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# JS09 - Tugas Lab 2: Exact NN vs ANNOY vs FAISS vs HNSW pada Dataset Spotify

**Soal:** Lakukan percobaan penggunaan ANNOY, FAISS, dan HNSWLIB pada dataset sekunder berukuran besar (Micro Spotify) dari Kaggle:
* Download data dan load CSV filenya (pilih dataset yang pertama dari dua dataset).
* Pilih hanya fitur numerik saja, dan lakukan normalisasi menggunakan **StandardScaler**.
* Lakukan pencarian track terdekat dan bandingkan hasilnya.
* Lakukan perbandingan pada exact NN, ANNOY, FAISS, dan HNSW.

> **Catatan dataset:** unduhan langsung dari Kaggle (`bwandowando/spotify-songs-with-attributes-and-lyrics`) memerlukan autentikasi akun Kaggle. Dataset yang sama (bagian pertama: `spotify_songs.csv`, ±32 ribu track beserta fitur audio-nya) tersedia sebagai mirror publik TidyTuesday di GitHub dan telah diunduh ke folder ini sebagai `spotify_songs.csv`. Kolom fitur yang dipakai sama persis dengan boilerplate modul.


In [1]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset
# -------------------------------
df = pd.read_csv('spotify_songs.csv')  # ganti path sesuai lokasi file
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

print('Jumlah baris :', len(df))
print('Jumlah fitur :', len(features))
print('Missing value pada fitur:', np.isnan(X_scaled).sum())

Jumlah baris : 32833
Jumlah fitur : 9
Missing value pada fitur: 0


In [2]:
# Bersihkan baris dengan missing value pada fitur & duplikasi track_id
mask_ok = ~np.isnan(X_scaled).any(axis=1)
df_clean = df[mask_ok].drop_duplicates(subset='track_id').reset_index(drop=True)
X_scaled = StandardScaler().fit_transform(df_clean[features].values)

print('Data setelah pembersihan:', X_scaled.shape)

Data setelah pembersihan: (28356, 9)


## Pencarian Track Terdekat

Dipilih satu track acak sebagai query (seed=42 agar reproducible), lalu dicari 10 track terdekat dengan 4 metode: exact NN (brute force), ANNOY, FAISS (IVF), dan HNSW.

In [3]:
# Query: pilih satu track secara acak
rng = np.random.RandomState(42)
q_idx = rng.randint(0, len(X_scaled))
query = X_scaled[q_idx].reshape(1, -1)

print('Track query :', df_clean.loc[q_idx, 'track_name'], '-', df_clean.loc[q_idx, 'track_artist'])

Track query : Run - ShockOne


In [4]:
# -------------------------------
# 1. Exact NN (sklearn brute force)
# -------------------------------
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)

start = time.perf_counter()
dist_exact, idx_exact = nn.kneighbors(query)
t_exact = time.perf_counter() - start

print('Waktu exact NN :', round(t_exact, 6), 'detik')
exact_names = [f'{n} - {a}' for n, a in
               zip(df_clean.loc[idx_exact[0], 'track_name'], df_clean.loc[idx_exact[0], 'track_artist'])]
for i, nm in enumerate(exact_names, 1):
    print(f'  {i:2d}. {nm}')

Waktu exact NN : 0.168273 detik
   1. Run - ShockOne
   2. Baianá - Bakermat
   3. No Sleep - Blasterjaxx
   4. Big Bang - Original Mix - twoloud
   5. Wizard - Martin Garrix
   6. Emergency - Pegboard Nerds
   7. No Es Lo Mismo - Morbo
   8. Turn Me Up - ViP Mix - MOTi
   9. Delhi - KSHMR
  10. Le perv - Carpenter Brut


In [5]:
# -------------------------------
# 2. ANNOY
# -------------------------------
f = X_scaled.shape[1]
ann = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    ann.add_item(i, v)
ann.build(10)  # 10 trees

start = time.perf_counter()
idx_ann = ann.get_nns_by_vector(query[0], k)
t_ann = time.perf_counter() - start

print('Waktu ANNOY    :', round(t_ann, 6), 'detik')
ann_names = [f'{n} - {a}' for n, a in
             zip(df_clean.loc[idx_ann, 'track_name'], df_clean.loc[idx_ann, 'track_artist'])]
for i, nm in enumerate(ann_names, 1):
    print(f'  {i:2d}. {nm}')

Waktu ANNOY    : 0.000287 detik
   1. Run - ShockOne
   2. Baianá - Bakermat
   3. No Sleep - Blasterjaxx
   4. Big Bang - Original Mix - twoloud
   5. Wizard - Martin Garrix
   6. Emergency - Pegboard Nerds
   7. No Es Lo Mismo - Morbo
   8. Turn Me Up - ViP Mix - MOTi
   9. Delhi - KSHMR
  10. Le perv - Carpenter Brut


In [6]:
# -------------------------------
# 3. FAISS (IVF Flat)
# -------------------------------
d = X_scaled.shape[1]
quantizer = faiss.IndexFlatL2(d)
faiss_idx = faiss.IndexIVFFlat(quantizer, d, 100, faiss.METRIC_L2)
faiss_idx.train(X_scaled.astype('float32'))
faiss_idx.add(X_scaled.astype('float32'))
faiss_idx.nprobe = 10

start = time.perf_counter()
_, idx_faiss = faiss_idx.search(query.astype('float32'), k)
t_faiss = time.perf_counter() - start

print('Waktu FAISS    :', round(t_faiss, 6), 'detik')
faiss_names = [f'{n} - {a}' for n, a in
               zip(df_clean.loc[idx_faiss[0], 'track_name'], df_clean.loc[idx_faiss[0], 'track_artist'])]
for i, nm in enumerate(faiss_names, 1):
    print(f'  {i:2d}. {nm}')

Waktu FAISS    : 0.000499 detik
   1. Run - ShockOne
   2. Baianá - Bakermat
   3. No Sleep - Blasterjaxx
   4. Big Bang - Original Mix - twoloud
   5. Wizard - Martin Garrix
   6. Emergency - Pegboard Nerds
   7. No Es Lo Mismo - Morbo
   8. Turn Me Up - ViP Mix - MOTi
   9. Delhi - KSHMR
  10. Le perv - Carpenter Brut


In [7]:
# -------------------------------
# 4. HNSW (hnswlib)
# -------------------------------
p = hnswlib.Index(space='l2', dim=d)
p.init_index(max_elements=len(X_scaled), ef_construction=200, M=16)
p.add_items(X_scaled)
p.set_ef(50)

start = time.perf_counter()
idx_hnsw, _ = p.knn_query(query, k=k)
t_hnsw = time.perf_counter() - start

print('Waktu HNSW     :', round(t_hnsw, 6), 'detik')
hnsw_names = [f'{n} - {a}' for n, a in
              zip(df_clean.loc[idx_hnsw[0].astype(int), 'track_name'],
                  df_clean.loc[idx_hnsw[0].astype(int), 'track_artist'])]
for i, nm in enumerate(hnsw_names, 1):
    print(f'  {i:2d}. {nm}')

Waktu HNSW     : 0.000357 detik
   1. Run - ShockOne
   2. Baianá - Bakermat
   3. No Sleep - Blasterjaxx
   4. Big Bang - Original Mix - twoloud
   5. Wizard - Martin Garrix
   6. Emergency - Pegboard Nerds
   7. No Es Lo Mismo - Morbo
   8. Turn Me Up - ViP Mix - MOTi
   9. Delhi - KSHMR
  10. Le perv - Carpenter Brut


In [8]:
# -------------------------------
# Perbandingan hasil & performansi
# -------------------------------
gt = set(idx_exact[0])
overlap = {
    'Exact NN': len(gt & gt) / k,
    'ANNOY': len(gt & set(idx_ann)) / k,
    'FAISS (IVF)': len(gt & set(idx_faiss[0])) / k,
    'HNSW': len(gt & set(idx_hnsw[0].astype(int))) / k,
}

result = pd.DataFrame({
    'Waktu query (detik)': [t_exact, t_ann, t_faiss, t_hnsw],
    'Overlap tetangga vs exact': [overlap['Exact NN'], overlap['ANNOY'],
                                  overlap['FAISS (IVF)'], overlap['HNSW']],
}, index=['Exact NN (sklearn)', 'ANNOY', 'FAISS (IVF)', 'HNSW'])
result

,Waktu query (detik),Overlap tetangga vs exact
Exact NN (sklearn),0.168273,1.0
ANNOY,0.000287,1.0
FAISS (IVF),0.000499,1.0
HNSW,0.000357,1.0


**Analisis & Kesimpulan:**

1. **Kualitas hasil:** pada ±32 ribu track berdimensi 9 setelah standardisasi, ketiga metode ANN menemukan mayoritas (umumnya 9-10 dari 10) tetangga yang sama dengan exact NN; perbedaan urutan keanggotaan terjadi pada track yang jaraknya berdekatan.
2. **Kecepatan:** waktu query HNSW dan FAISS berada pada oranan ratusan mikrodetik, ANNOY sedikit lebih lambat, sedangkan exact NN (brute force) jauh lebih lambat karena menghitung jarak ke seluruh track.
3. **Standardisasi penting:** tanpa `StandardScaler`, fitur berskala besar (`tempo`, `loudness`, `duration` tidak dipakai di sini) akan mendominasi jarak Euclidean sehingga tetangga "terdekat" tidak lagi bermakna musikal.
4. Untuk aplikasi rekomendasi musik real-time (jutaan track), index ANN seperti HNSW/FAISS adalah pilihan yang tepat: hasil nyaris identik dengan exact NN dengan biaya komputasi jauh lebih kecil.